# Table 3 — Full Results Viewer
Reads results directly from the **saved outputs** of `000Final_ml-Articlev3.ipynb`.  
**No rerun required.** Just set `NB_PATH` and run all cells.

In [1]:
import json, re, numpy as np, pandas as pd
from pathlib import Path

# ── CHANGE THIS to wherever your notebook is saved ──────────────────────────
NB_PATH = Path(r"C:\Users\asus\Article-10-06-2026-ML-portfoliov1\000Final_ml-Articlev3.ipynb")
# ────────────────────────────────────────────────────────────────────────────

with open(NB_PATH, encoding='utf-8') as f:
    nb = json.load(f)

# Collect all printed output text, keyed by execution_count
outputs = {}
for cell in nb['cells']:
    if cell['cell_type'] != 'code': continue
    n = cell.get('execution_count')
    text = ''.join(t for o in cell.get('outputs', []) for t in o.get('text', []))
    if n and text:
        outputs[n] = text

print('Notebook loaded. Executed cells found:', sorted(outputs.keys()))

Notebook loaded. Executed cells found: [1, 2, 3, 4, 5, 7, 8, 9, 10, 12, 13, 14, 15, 17, 18, 19, 21, 23, 24]


In [2]:
TICKERS = ['ASBN','CCBN','HSBK','KCEL','KEGC','KZTK','KZTO','KZAP']

# ── Parse classical ML results (cell 12) ────────────────────────────────────
ML_PAT = re.compile(
    r'(RandomForest|XGBoost|LightGBM|SVR|ElasticNet|KNN)\s*: '
    r'RMSE=([0-9.]+) \| R2=([+-]?[0-9.]+) \| DA=([0-9.]+) '
    r'\(z=([+-][0-9.]+), p=([0-9.]+)\)\s*(\*?)'
)
ml_rows, ticker = [], None
for line in outputs[12].splitlines():
    m = re.match(r'Training: (\w+)', line)
    if m: ticker = m.group(1)
    m = ML_PAT.search(line)
    if m and ticker:
        model, rmse, r2, da, z, p, sig = m.groups()
        ml_rows.append(dict(Ticker=ticker, Model=model,
                            RMSE=float(rmse), R2=float(r2), DA=float(da),
                            DA_z=float(z), DA_pval=float(p), DA_sig=sig=='*'))
ml_df = pd.DataFrame(ml_rows)

# ── Parse DL results (cell 14) ──────────────────────────────────────────────
DL_PAT = re.compile(
    r'(LSTM|CNN)\s+: RMSE=([0-9.]+) \| R2=([+-]?[0-9.]+) \| DA=([0-9.]+) '
    r'\(z=([+-][0-9.]+), p=([0-9.]+)\)\s*(\*?)'
)
dl_rows, ticker = [], None
for line in outputs[14].splitlines():
    m = re.match(r'DL training: (\w+)', line)
    if m: ticker = m.group(1)
    m = DL_PAT.search(line)
    if m and ticker:
        model, rmse, r2, da, z, p, sig = m.groups()
        dl_rows.append(dict(Ticker=ticker, Model=model,
                            RMSE=float(rmse), R2=float(r2), DA=float(da),
                            DA_z=float(z), DA_pval=float(p), DA_sig=sig=='*'))
dl_df = pd.DataFrame(dl_rows)

all_df = pd.concat([ml_df, dl_df], ignore_index=True)
print(f'Parsed: {len(ml_df)} classical rows  |  {len(dl_df)} DL rows')

Parsed: 48 classical rows  |  16 DL rows


In [3]:
# ── TABLE 3 — 8-model aggregate summary ─────────────────────────────────────
agg = (
    all_df.groupby('Model')
    .agg(
        Avg_RMSE=('RMSE', 'mean'),
        Avg_R2  =('R2',   'mean'),
        Avg_DA  =('DA',   'mean'),
        Sig_Count=('DA_sig', 'sum'),
    )
    .round({'Avg_RMSE':4, 'Avg_R2':4, 'Avg_DA':4})
    .sort_values('Avg_DA', ascending=False)
)
agg['Sig_Count'] = agg['Sig_Count'].astype(int)
agg['Note'] = agg.index.map(lambda m: '3 folds [!]' if m in ('LSTM','CNN') else '4 folds')
# REVIEWER NOTE (Point 5): DL uses 3 folds vs 4 for classical ML.
# Cross-model rank comparisons involving DL vs classical must be flagged.
# [!] = fold asymmetry — interpret cross-class rankings with caution.

print('Table 3. Out-of-Sample Performance — All 8 Models (averaged across 8 KASE equities)')
display(agg)

Table 3. Out-of-Sample Performance — All 8 Models (averaged across 8 KASE equities)


,Avg_RMSE,Avg_R2,Avg_DA,Sig_Count,Note
Model,,,,,
ElasticNet,0.0963,-0.1492,0.5842,6,4 folds
RandomForest,0.0975,-0.2097,0.5514,4,4 folds
KNN,0.0984,-0.2300,0.5495,3,4 folds
LightGBM,0.1010,-0.2967,0.5396,4,4 folds
LSTM,0.1245,-1.4656,0.5281,2,* 3 folds
XGBoost,0.1015,-0.3026,0.5275,3,4 folds
SVR,0.1016,-0.3179,0.5234,2,4 folds
CNN,0.4454,-47.8560,0.4675,1,* 3 folds


In [4]:
# ── TABLE 3b — Per-ticker LSTM and CNN ──────────────────────────────────────
dl_pivot = dl_df.copy()
dl_pivot['DA_sig'] = dl_pivot['DA_sig'].map({True:'Yes*', False:''})
dl_pivot = dl_pivot.set_index(['Ticker','Model']).unstack('Model')
dl_pivot.columns = [f'{m}_{c}' for c, m in dl_pivot.columns]
dl_pivot = dl_pivot.reindex(TICKERS)

print('Table 3b. Per-Ticker LSTM and CNN Walk-Forward Results')
print('(3 expanding folds, gap = 21 days; Yes* = p < 0.05 one-tailed z-test)')
display(dl_pivot[[
    'LSTM_RMSE','LSTM_R2','LSTM_DA','LSTM_DA_z','LSTM_DA_pval','LSTM_DA_sig',
    'CNN_RMSE', 'CNN_R2', 'CNN_DA', 'CNN_DA_z', 'CNN_DA_pval', 'CNN_DA_sig'
]])

Table 3b. Per-Ticker LSTM and CNN Walk-Forward Results
(3 expanding folds, gap = 21 days; Yes* = p < 0.05 one-tailed z-test)


,LSTM_RMSE,LSTM_R2,LSTM_DA,LSTM_DA_z,LSTM_DA_pval,LSTM_DA_sig,CNN_RMSE,CNN_R2,CNN_DA,CNN_DA_z,CNN_DA_pval,CNN_DA_sig
Ticker,,,,,,,,,,,,
ASBN,0.22287,-0.4841,0.480,-0.90,0.816,,1.05297,-32.1278,0.468,-1.44,0.925,
CCBN,0.15156,-0.9769,0.616,5.14,0.000,Yes*,0.52432,-22.6596,0.327,-7.66,1.000,
HSBK,0.07730,-0.2822,0.705,9.11,0.000,Yes*,0.41725,-36.3623,0.498,-0.09,0.536,
KCEL,0.11416,-2.5726,0.482,-0.81,0.791,,0.42115,-47.6245,0.545,1.98,0.024,Yes*
KEGC,0.05885,-3.1588,0.512,0.54,0.294,,0.42050,-211.3068,0.520,0.90,0.184,
KZTK,0.15069,-1.5416,0.453,-2.07,0.981,,0.20679,-3.7860,0.425,-3.34,1.000,
KZTO,0.09907,-1.8430,0.449,-2.25,0.988,,0.28896,-23.1854,0.457,-1.89,0.971,
KZAP,0.12113,-0.8653,0.528,1.26,0.103,,0.23119,-5.7953,0.500,0.00,0.500,
